# ObserveOps Log Analytics — batch ETL in PySpark

The same pipeline as `log_pipeline.py`, one step at a time. Run `python generate_logs.py` first so `data/raw/` has files.

In [ ]:
import os, sys, glob
# Make Spark's Python libs importable from a plain kernel (spark-submit does this for scripts).
_sh = os.environ.get("SPARK_HOME", "/usr/local/spark")
sys.path[:0] = [f"{_sh}/python"] + glob.glob(f"{_sh}/python/lib/py4j-*-src.zip")

from pyspark.sql import SparkSession
from log_pipeline import extract, transform, aggregate, load

spark = (SparkSession.builder.appName("observeops-log-analytics-nb")
         .config("spark.sql.sources.partitionOverwriteMode", "dynamic")
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")

## Extract — read with an explicit schema
Nothing executes yet. `read` is a transformation: Spark only records the plan.

In [ ]:
raw = extract(spark)
raw.printSchema()
raw.show(5, truncate=False)   # show() is an action — this is the first time anything runs

## Transform — two data-quality gates
1. Required fields present. 2. Dedupe by `request_id` so a retried request counts once.
The reconciliation below is how you *know* what the gates removed.

In [ ]:
typed, clean, events = transform(raw)
n_raw, n_clean, n_events = raw.count(), clean.count(), events.count()
print(f"raw rows         : {n_raw}")
print(f"after validation : {n_clean}   (dropped {n_raw - n_clean} malformed)")
print(f"after dedupe     : {n_events}   (dropped {n_clean - n_events} duplicates)")

## Aggregate — each `groupBy` is a shuffle
Rows with the same key have to travel to the same executor. That network move is the expensive part of a Spark job.

In [ ]:
per_service_day, per_endpoint, per_minute = aggregate(events)
per_service_day.orderBy("date", "service").show(20, truncate=False)
per_endpoint.orderBy("date", "service", "path").show(20, truncate=False)

## Load — Parquet partitioned by date
Dynamic partition overwrite means re-running one day rewrites only that day's partition: an idempotent load.

In [ ]:
load("per_service_day", per_service_day)
load("per_endpoint", per_endpoint)
load("per_minute", per_minute)

# read one partitioned output back — Spark uses the date=... folder names as a column
spark.read.parquet("data/curated/per_service_day").orderBy("date", "service").show(20, truncate=False)

## Batch vs streaming
`stream_pipeline.py` runs the same aggregation with `readStream` over the folder as an unbounded source, a 10-minute watermark for late events, and a checkpoint so a restart resumes instead of reprocessing. Same DataFrame code — different source semantics.